In [1]:
# 01 — Thư viện, dữ liệu, cấu hình
from pathlib import Path
from itertools import combinations
import hashlib
import json
import random
import copy
import platform
import time

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

PROJECT = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
DATA_CANDIDATES = [
    PROJECT / '03_model_ready_development.csv',
    PROJECT / 'outputs/03_feature_engineering/03_model_ready_development.csv',
    Path('03_model_ready_development.csv'),
    Path('outputs/03_feature_engineering/03_model_ready_development.csv'),
]
INPUT_PATH = next((p for p in DATA_CANDIDATES if p.is_file()), DATA_CANDIDATES[0])
OUTPUT_DIR = PROJECT / 'outputs/06_2_ae_validation_A'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURES = ['z_gap', 'z_intraday', 'z_range', 'z_volume']
SEEDS = [7, 19, 42, 73, 101]
LR_MAIN = 0.001
LR_ALTERNATIVES = [0.0005, 0.002]  # Thử có mục đích, chưa chọn tối ưu
MAX_EPOCHS = 1500
PATIENCE = 80
MIN_DELTA = 1e-5
BATCH_SIZE = 512
TOP_RATES = [0.01, 0.05]  # Để so độ nhất quán, KHÔNG phải ngưỡng cảnh báo

if not INPUT_PATH.is_file():
    raise FileNotFoundError(f'Không có file đầu vào: {INPUT_PATH.resolve()}')
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
print('Nguồn:', INPUT_PATH.resolve())
print('Đầu ra:', OUTPUT_DIR.resolve())

Nguồn: D:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks\outputs\03_feature_engineering\03_model_ready_development.csv
Đầu ra: D:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks\outputs\06_2_ae_validation_A


In [2]:
# 02 — Đọc đúng Train, Validation select; loại hoàn toàn val calibrate và Test
s = pd.read_csv(INPUT_PATH, encoding='utf-8-sig')
need = ['ticker', 'date', 'split', 'val_role', *FEATURES]
assert set(need).issubset(s.columns), f'Thiếu: {set(need) - set(s.columns)}'
assert s[need].notna().all().all(), 'Thiếu giá trị'
assert not s.duplicated(['ticker','date']).any(), 'Trùng khóa'
assert s['split'].isin(['train','val']).all(), 'File Development có Test!'
assert s.loc[s.split.eq('val'),'val_role'].isin(['select','calibrate']).all()
assert s.loc[s.split.eq('train'),'val_role'].eq('not_applicable').all()
s['date'] = pd.to_datetime(s['date'], format='%Y-%m-%d', errors='raise')
for f in FEATURES: s[f] = pd.to_numeric(s[f], errors='raise')
assert np.isfinite(s[FEATURES].to_numpy(float)).all()
train = s.loc[s.split.eq('train'),need].sort_values(['date','ticker']).reset_index(drop=True)
val = s.loc[(s.split.eq('val')) & (s.val_role.eq('select')),need].sort_values(['date','ticker']).reset_index(drop=True)
del s
assert len(train) == 10387 and len(val) == 1430, 'Sai số mẫu so với Bước 05 đã chốt'
assert train.date.max() < val.date.min()
X_train = train[FEATURES].to_numpy(np.float32)
X_val = val[FEATURES].to_numpy(np.float32)
train_tensor = torch.from_numpy(X_train)
val_tensor = torch.from_numpy(X_val)
print(f'Train {len(train)}, Val-select {len(val)}; số chiều {X_train.shape[1]}')

Train 10387, Val-select 1430; số chiều 4


In [11]:
class SmallAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4, 2), nn.Tanh(), nn.Linear(2, 4))
    def forward(self, x):
        return self.net(x)

@torch.no_grad()
def recon_err(model, x_tensor):
    model.eval()
    return (model(x_tensor) - x_tensor).square().mean(dim=1).cpu().numpy().astype('float64')

old_path = PROJECT / r'D:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks\outputs\05_model_training_A\05_ae.pt'
old_scores_path = PROJECT / r'D:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks\outputs\05_model_training_A\05_val_select_scores.csv'
reproduction = {'previous_checkpoint_found': old_path.is_file(),
                'previous_scores_found': old_scores_path.is_file()}
if old_path.is_file():
    old = torch.load(old_path, map_location='cpu', weights_only=True)
    old_model = SmallAE(); old_model.load_state_dict(old['state_dict'])
    old_val_scores = recon_err(old_model, val_tensor)
    reproduction['recomputed_val_mse'] = float(old_val_scores.mean())
    if old_scores_path.is_file():
        ref = pd.read_csv(old_scores_path, encoding='utf-8-sig')
        xref = val[['ticker','date']].copy(); xref['date'] = xref.date.dt.strftime('%Y-%m-%d')
        aligned = xref.merge(ref[['ticker','date','score_ae']],on=['ticker','date'],how='left',validate='1:1')
        assert aligned.score_ae.notna().all()
        reproduction['max_abs_score_diff'] = float(np.abs(old_val_scores-aligned.score_ae.to_numpy()).max())
        if reproduction['max_abs_score_diff'] > 1e-4:
            raise ValueError('Checkpoint Bước 05 không khớp điểm đã công bố')
print('Đối chiếu Bước 05:', reproduction)

Đối chiếu Bước 05: {'previous_checkpoint_found': True, 'previous_scores_found': True, 'recomputed_val_mse': 1.2263565243163204, 'max_abs_score_diff': 1.943359379197318e-06}


In [12]:
def run_ae(seed, lr, max_epochs=MAX_EPOCHS):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = SmallAE()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(train_tensor), batch_size=BATCH_SIZE,
                        shuffle=True, generator=torch.Generator().manual_seed(seed), num_workers=0)
    criterion = nn.MSELoss()
    best_val = float('inf'); best_epoch = 0; best_state = None
    no_improve = 0; rows = []; checkpoints = {}
    stop_reason = 'max_epochs'
    start = time.perf_counter()
    for epoch in range(1, max_epochs + 1):
        model.train()
        for (xb,) in loader:
            optimizer.zero_grad(set_to_none=True)
            y = model(xb)
            loss = criterion(y, xb)
            if not torch.isfinite(loss):
                raise RuntimeError(f'Loss không hữu hạn: seed={seed}, lr={lr}, epoch={epoch}')
            loss.backward(); optimizer.step()
        model.eval()
        with torch.no_grad():
            train_mse = float(criterion(model(train_tensor),train_tensor).item())
            val_mse = float(criterion(model(val_tensor),val_tensor).item())
        if not np.isfinite([train_mse,val_mse]).all():
            raise RuntimeError('MSE không hữu hạn')
        rows.append({'epoch':epoch, 'train_mse':train_mse, 'val_select_mse':val_mse})
        if epoch in (300,1000,1500) and seed==42 and lr==LR_MAIN:
            checkpoints[epoch] = recon_err(model, val_tensor)
        if val_mse < best_val - MIN_DELTA:
            best_val = val_mse; best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict()); no_improve = 0
        else: no_improve += 1
        if no_improve >= PATIENCE:
            stop_reason = 'early_stopping'
            break
    model.load_state_dict(best_state); model.eval()
    scores_val = recon_err(model, val_tensor)
    scores_train = recon_err(model, train_tensor)
    # MSE toàn tập sau khi khôi phục checkpoint tốt nhất
    stats = dict(seed=seed, lr=lr, max_epochs=max_epochs, epochs_executed=epoch,
                 best_epoch=best_epoch, best_val_mse=best_val,
                 restored_val_mse=float(scores_val.mean()),
                 restored_train_mse=float(scores_train.mean()),
                 stop_reason=stop_reason, seconds=float(time.perf_counter()-start),
                 best_epoch_close_to_limit=bool(best_epoch >= max_epochs - 50))
    return stats, pd.DataFrame(rows), scores_val, scores_train, checkpoints, copy.deepcopy(best_state)

experiments = [(seed, LR_MAIN) for seed in SEEDS] + [(42,lr) for lr in LR_ALTERNATIVES]
run_rows = []; learning_curves = []; scores_by_run = {}; train_scores_by_run = {}
checkpoint_scores = {}; final_states = {}
for i,(seed,lr) in enumerate(experiments,1):
    print(f'Run {i}/{len(experiments)}: seed={seed}, lr={lr}, max_epochs={MAX_EPOCHS}',flush=True)
    stats,curve,sv,st,cp,weights = run_ae(seed,lr)
    label = f'lr{lr:g}_seed{seed}'
    run_rows.append(stats)
    curve['seed']=seed; curve['learning_rate']=lr
    learning_curves.append(curve)
    scores_by_run[label]=sv
    train_scores_by_run[label]=st
    if seed==42 and lr==LR_MAIN:
        checkpoint_scores=cp; final_states=weights
    print(f"  best_epoch {stats['best_epoch']}, executed {stats['epochs_executed']}, "
          f"val_MSE {stats['best_val_mse']:.5f}, stop {stats['stop_reason']}, seconds {stats['seconds']:.1f}",flush=True)

runs = pd.DataFrame(run_rows)
curves = pd.concat(learning_curves,ignore_index=True)
runs.to_csv(OUTPUT_DIR/'06_2_ae_run_summary.csv',index=False,encoding='utf-8-sig')
curves.to_csv(OUTPUT_DIR/'06_2_ae_training_histories.csv',index=False,encoding='utf-8-sig')

Run 1/7: seed=7, lr=0.001, max_epochs=1500


  best_epoch 1500, executed 1500, val_MSE 1.03623, stop max_epochs, seconds 302.7
Run 2/7: seed=19, lr=0.001, max_epochs=1500
  best_epoch 1500, executed 1500, val_MSE 1.04008, stop max_epochs, seconds 181.4
Run 3/7: seed=42, lr=0.001, max_epochs=1500
  best_epoch 1495, executed 1500, val_MSE 1.03729, stop max_epochs, seconds 180.8
Run 4/7: seed=73, lr=0.001, max_epochs=1500
  best_epoch 1491, executed 1500, val_MSE 1.03909, stop max_epochs, seconds 177.9
Run 5/7: seed=101, lr=0.001, max_epochs=1500
  best_epoch 1496, executed 1500, val_MSE 1.03835, stop max_epochs, seconds 180.8
Run 6/7: seed=42, lr=0.0005, max_epochs=1500
  best_epoch 1492, executed 1500, val_MSE 1.44830, stop max_epochs, seconds 323.1
Run 7/7: seed=42, lr=0.002, max_epochs=1500
  best_epoch 1499, executed 1500, val_MSE 0.93143, stop max_epochs, seconds 314.4


In [13]:
def top_ids(a, n):
    return set(np.argsort(-np.asarray(a),kind='stable')[:n].tolist())

pairs=[]
for a,b in combinations(SEEDS,2):
    sa=scores_by_run[f'lr{LR_MAIN:g}_seed{a}']; sb=scores_by_run[f'lr{LR_MAIN:g}_seed{b}']
    r={'seed_a':a, 'seed_b':b, 'spearman':float(spearmanr(sa,sb).statistic)}
    for p in TOP_RATES:
        n=int(np.ceil(len(val)*p))
        r[f'top{int(p*100)}_overlap_pct']=100*len(top_ids(sa,n)&top_ids(sb,n))/n
    pairs.append(r)
pairs=pd.DataFrame(pairs)
pairs.to_csv(OUTPUT_DIR/'06_2_ae_seed_pairwise.csv',index=False,encoding='utf-8-sig')
print('\nĐộ ổn định giữa 5 seed, LR=0.001:')
print(pairs[['spearman','top1_overlap_pct','top5_overlap_pct']].agg(['median','min','max']))


Độ ổn định giữa 5 seed, LR=0.001:
        spearman  top1_overlap_pct  top5_overlap_pct
median  0.999895             100.0         98.611111
min     0.999675             100.0         98.611111
max     0.999972             100.0        100.000000


In [14]:
# 06 — Ảnh hưởng của điểm gap cực trị, lỗi theo đặc trưng và checkpoint
reference_label = f'lr{LR_MAIN:g}_seed42'
ref_train_scores=train_scores_by_run[reference_label]
ref_val_scores=scores_by_run[reference_label]
ref_model = SmallAE(); ref_model.load_state_dict(final_states); ref_model.eval()
with torch.no_grad():
    sq_train=(ref_model(train_tensor)-train_tensor).square().numpy().astype(float)
    sq_val=(ref_model(val_tensor)-val_tensor).square().numpy().astype(float)

extreme_mask = np.abs(X_train[:,0])>30
contribution = float(ref_train_scores[extreme_mask].sum()/ref_train_scores.sum()*100)
loss_components=[]
for split, sq in [('train',sq_train),('val_select',sq_val)]:
    totals=sq.sum(axis=0)
    for j,name in enumerate(FEATURES):
        loss_components.append(dict(subset=split,feature=name,mse=float(sq[:,j].mean()),
                                  share_squared_error_pct=float(100*totals[j]/totals.sum())))
loss_components=pd.DataFrame(loss_components)
loss_components.to_csv(OUTPUT_DIR/'06_2_ae_error_by_feature.csv',index=False,encoding='utf-8-sig')

checkpoint_rows=[]
for ep,scores in sorted(checkpoint_scores.items()):
    n=int(np.ceil(len(val)*0.05))
    checkpoint_rows.append(dict(epoch=ep,val_mse=float(np.mean(scores)),
       spearman_vs_best=float(spearmanr(scores,ref_val_scores).statistic),
       top5_overlap_vs_best_pct=100*len(top_ids(scores,n)&top_ids(ref_val_scores,n))/n))
checkpoints=pd.DataFrame(checkpoint_rows)
checkpoints.to_csv(OUTPUT_DIR/'06_2_ae_checkpoint_comparison.csv',index=False,encoding='utf-8-sig')

# Các thay đổi learning rate cũng không tự xác lập chất lượng phát hiện
lr_rows=[]
for lr in [LR_MAIN]+LR_ALTERNATIVES:
    scores=scores_by_run[f'lr{lr:g}_seed42'];n=int(np.ceil(len(val)*0.05))
    lr_rows.append(dict(learning_rate=lr, val_mse=float(np.mean(scores)),
       spearman_vs_lr_main=float(spearmanr(scores,ref_val_scores).statistic),
       top5_overlap_vs_lr_main_pct=100*len(top_ids(scores,n)&top_ids(ref_val_scores,n))/n))
lr_cmp=pd.DataFrame(lr_rows)
lr_cmp.to_csv(OUTPUT_DIR/'06_2_ae_lr_sensitivity.csv',index=False,encoding='utf-8-sig')

In [15]:
vmeta=val[['ticker','date']].copy()
vmeta['date']=vmeta.date.dt.strftime('%Y-%m-%d')
vmeta['score_ae']=ref_val_scores
vmeta['month']=vmeta['date'].str[:7]
vmeta['rank']=vmeta.score_ae.rank(method='first',ascending=False).astype(int)
vmeta.to_csv(OUTPUT_DIR/'06_2_ae_val_select_scores_seed42.csv',index=False,encoding='utf-8-sig')
summary_ticker=vmeta.groupby('ticker').agg(n=('score_ae','size'),median_score=('score_ae','median'),p99_score=('score_ae',lambda a:a.quantile(.99))).reset_index()
month_rows=[]
for p in TOP_RATES:
    n=int(np.ceil(len(vmeta)*p))
    block=vmeta.nsmallest(n,'rank')
    by_month=block.groupby('month').size()
    for month, group in vmeta.groupby('month'):
        month_rows.append(dict(top_rate=p,month=month,val_rows=len(group),top_n=n,
             top_rows_in_month=int(by_month.get(month,0)),top_share_pct=100*by_month.get(month,0)/n))
summary_month=pd.DataFrame(month_rows)
summary_ticker.to_csv(OUTPUT_DIR/'06_2_ae_by_ticker.csv',index=False,encoding='utf-8-sig')
summary_month.to_csv(OUTPUT_DIR/'06_2_ae_by_month.csv',index=False,encoding='utf-8-sig')

In [16]:
vmeta=val[['ticker','date']].copy()
vmeta['date']=vmeta.date.dt.strftime('%Y-%m-%d')
vmeta['score_ae']=ref_val_scores
vmeta['month']=vmeta['date'].str[:7]
vmeta['rank']=vmeta.score_ae.rank(method='first',ascending=False).astype(int)
vmeta.to_csv(OUTPUT_DIR/'06_2_ae_val_select_scores_seed42.csv',index=False,encoding='utf-8-sig')
summary_ticker=vmeta.groupby('ticker').agg(n=('score_ae','size'),median_score=('score_ae','median'),p99_score=('score_ae',lambda a:a.quantile(.99))).reset_index()
month_rows=[]
for p in TOP_RATES:
    n=int(np.ceil(len(vmeta)*p))
    block=vmeta.nsmallest(n,'rank')
    by_month=block.groupby('month').size()
    for month, group in vmeta.groupby('month'):
        month_rows.append(dict(top_rate=p,month=month,val_rows=len(group),top_n=n,
             top_rows_in_month=int(by_month.get(month,0)),top_share_pct=100*by_month.get(month,0)/n))
summary_month=pd.DataFrame(month_rows)
summary_ticker.to_csv(OUTPUT_DIR/'06_2_ae_by_ticker.csv',index=False,encoding='utf-8-sig')
summary_month.to_csv(OUTPUT_DIR/'06_2_ae_by_month.csv',index=False,encoding='utf-8-sig')

In [17]:
# 08 — Biểu đồ và báo cáo
fig,ax=plt.subplots(figsize=(10,5.4))
for seed in SEEDS:
    h=curves[(curves.seed==seed)&(np.isclose(curves.learning_rate,LR_MAIN))]
    ax.plot(h.epoch,h.val_select_mse,label=f'Seed {seed}',linewidth=1.2)
ax.set(xlabel='Epoch',ylabel='MSE on Validation select',title='AE 4-2-4 A: Validation reconstruction MSE across seeds (lr=0.001)')
ax.legend(ncol=3,fontsize=8);ax.grid(alpha=.2)
fig.tight_layout();fig.savefig(OUTPUT_DIR/'06_2_ae_curves_by_seed.png',dpi=160);plt.close(fig)

config=dict(step='06_2_AE_validation_A',main_model='IF',model_under_test='AE (comparison)',
 input_sha256=hashlib.sha256(INPUT_PATH.read_bytes()).hexdigest(),
 train_n=len(train),val_select_n=len(val),features=FEATURES,
 architecture='4-2-4 (Tanh hidden, linear output)',optimizer='Adam',loss='MSE',
 seeds=SEEDS,lr_main=LR_MAIN,lr_alternatives=LR_ALTERNATIVES,max_epochs=MAX_EPOCHS,
 batch_size=BATCH_SIZE,patience=PATIENCE,min_delta=MIN_DELTA,
 floor_and_W='Unchanged from Step 03: W=60, train-only floor',
 test_policy='Test not accessed; Val-calibrate metadata filtered out before modeling; no threshold or proxy labels',
 reproduction=reproduction,torch=torch.__version__,python=platform.python_version())
(OUTPUT_DIR/'06_2_ae_config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
summary_lines=[
 '# Bước 06.2 — Kiểm chứng Autoencoder, phương án A',
 '',
 '**Vai trò:** AE là mô hình đối chiếu; Isolation Forest vẫn là mô hình chính.',
 f'- Train: **{len(train):,}**; Validation chọn cấu hình: **{len(val):,}**; không dùng Validation hiệu chỉnh hay Test.',
 '- Đầu vào 4 đặc trưng W=60, không bỏ phiên gap cực trị, không thêm scaler hay chuyển B.',
 f'- Kiểm tra lại checkpoint Bước 05: {reproduction}.',
 '',
 '## Bảng chạy',
 '```',runs.round(5).to_string(index=False),'```','',
 '## Độ ổn định giữa seed (LR=0.001, checkpoint có Val-MSE thấp nhất)',
 '```',pairs[['spearman','top1_overlap_pct','top5_overlap_pct']].agg(['median','min','max']).round(4).to_string(),'```','',
 f'- Số phiên Train |z_gap|>30: **{int(extreme_mask.sum())}**; chiếm **{contribution:.2f}%** tổng lỗi tái tạo Train tại checkpoint seed42.',
 '## Tỷ trọng lỗi theo đặc trưng',
 '```',loss_components.round(3).to_string(index=False),'```',
 '## Độ nhạy learning rate cùng seed 42',
 '```',lr_cmp.round(4).to_string(index=False),'```',
 '## So checkpoint cùng một lần học (seed 42)',
 '```',checkpoints.round(4).to_string(index=False),'```','',
 '## Các giới hạn trước khi kết luận',
 '- MSE nhỏ hơn không chứng minh khả năng phát hiện bất thường tốt hơn; Validation select không có nhãn thật.',
 '- Best checkpoint lựa chọn trên chính Validation select, nên các kết quả thứ hạng tại đây là phân tích phát triển, không phải đánh giá độc lập.',
 '- Nếu nhiều seed còn chạm giới hạn epoch, phải ghi nhận chưa có bằng chứng hội tụ, không tự tuyên bố đã tối ưu.',
 '- Xem độ ổn định Top-k và biến động giữa checkpoint; Top-k không phải ngưỡng cảnh báo.',
 '- Chưa dùng proxy, Validation calibrate hay Test và không chọn phương án B.',
 '- Sau bước này cần kiểm tra LOF 06.3, rồi chung OHLCV 06.4, cuối cùng mới so sánh ba mô hình ở 06.5.',
]
(OUTPUT_DIR/'06_2_ae_report.md').write_text('\n'.join(summary_lines),encoding='utf-8')
print('\n=== TÓM TẮT BƯỚC 06.2 ===')
print(runs[['seed','lr','epochs_executed','best_epoch','best_val_mse','stop_reason']].to_string(index=False))
print('Extreme Gap train loss contribution:',f'{contribution:.2f}%')
print('Outputs:', OUTPUT_DIR.resolve())


=== TÓM TẮT BƯỚC 06.2 ===
 seed     lr  epochs_executed  best_epoch  best_val_mse stop_reason
    7 0.0010             1500        1500      1.036229  max_epochs
   19 0.0010             1500        1500      1.040076  max_epochs
   42 0.0010             1500        1495      1.037287  max_epochs
   73 0.0010             1500        1491      1.039094  max_epochs
  101 0.0010             1500        1496      1.038348  max_epochs
   42 0.0005             1500        1492      1.448299  max_epochs
   42 0.0020             1500        1499      0.931429  max_epochs
Extreme Gap train loss contribution: 64.33%
Outputs: D:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks\outputs\06_2_ae_validation_A


In [18]:
tail_rows=[]
for (seed,lr),group in curves.groupby(['seed','learning_rate']):
    h=group.sort_values('epoch').tail(100)
    tail_rows.append({
        'seed':int(seed), 'lr':float(lr),
        'last100_val_mean':float(h.val_select_mse.mean()),
        'last100_val_std':float(h.val_select_mse.std()),
        'last100_val_min':float(h.val_select_mse.min()),
        'last100_val_max':float(h.val_select_mse.max()),
        'last_val_mse':float(h.val_select_mse.iloc[-1]),
    })
tail_stats=pd.DataFrame(tail_rows)
tail_stats.to_csv(OUTPUT_DIR/'06_2_ae_last100_stability.csv',index=False,encoding='utf-8-sig')
with (OUTPUT_DIR/'06_2_ae_report.md').open('a',encoding='utf-8') as f:
    f.write('\n## Độ dao động 100 epoch cuối (không phải độ chính xác)\n```\n')
    f.write(tail_stats.round(5).to_string(index=False))
    f.write('\n```\n')
    f.write('- Chú ý: learning rate cao có thể có MSE checkpoint thấp nhất nhưng dao động lớn hơn ở những epoch cuối; không chọn LR chỉ vì best MSE.\n')
print('Độ dao động 100 epoch cuối:')
print(tail_stats.round(5).to_string(index=False))

Độ dao động 100 epoch cuối:
 seed     lr  last100_val_mean  last100_val_std  last100_val_min  last100_val_max  last_val_mse
    7 0.0010           1.05016          0.00645          1.03623          1.06834       1.03623
   19 0.0010           1.05559          0.00765          1.04008          1.07591       1.04008
   42 0.0005           1.48105          0.02071          1.44830          1.52923       1.44990
   42 0.0010           1.04848          0.00649          1.03729          1.06711       1.03881
   42 0.0020           0.96231          0.02941          0.93143          1.08797       1.03792
   73 0.0010           1.05066          0.00609          1.03909          1.06326       1.04173
  101 0.0010           1.05134          0.00675          1.03835          1.06201       1.04025


In [19]:

# ==================================================
# BƯỚC 06.2 - LƯU CHECKPOINT AUTOENCODER TỐT NHẤT
# Seed 42 - Learning rate 0.001
# ==================================================

from pathlib import Path
import numpy as np
import torch

# 1. Kiểm tra các biến còn trong Jupyter
required = [
    "final_states",
    "SmallAE",
    "recon_err",
    "val_tensor",
    "scores_by_run",
    "runs",
    "OUTPUT_DIR",
    "FEATURES"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        f"Thiếu biến {missing}. "
        "Cần chạy ô huấn luyện AE trước khi lưu."
    )

assert len(final_states) > 0, "Chưa có trọng số AE"

# 2. Xác định checkpoint tham chiếu
selected = runs.loc[
    (runs["seed"] == 42)
    & np.isclose(runs["lr"], 0.001)
].iloc[0]

best_epoch = int(selected["best_epoch"])

print("Seed:", 42)
print("Learning rate:", 0.001)
print("Best epoch:", best_epoch)
print("Best Validation MSE:", selected["best_val_mse"])

# 3. Kiểm tra trọng số có đúng với điểm đã lưu
model = SmallAE()
model.load_state_dict(final_states)
model.eval()

scores = recon_err(model, val_tensor)
reference = scores_by_run["lr0.001_seed42"]

assert np.allclose(
    scores,
    reference,
    atol=1e-5,
    rtol=0
), "Trọng số không khớp điểm Validation!"

# 4. Lưu mô hình thành file .pt
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = output_dir / (
    f"06_2_ae_seed42_lr001_epoch{best_epoch}_best.pt"
)

# Không tự ghi đè checkpoint đã có
if checkpoint_path.exists():
    raise FileExistsError(
        f"File đã tồn tại: {checkpoint_path}"
    )

torch.save(
    {
        "state_dict": final_states,
        "architecture": "4-2-4",
        "features": list(FEATURES),
        "seed": 42,
        "learning_rate": 0.001,
        "best_epoch": best_epoch,
        "best_val_mse": float(selected["best_val_mse"])
    },
    checkpoint_path
)

# 5. TẢI LẠI ĐỂ XÁC MINH FILE ĐÃ LƯU
saved = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

restored_model = SmallAE()
restored_model.load_state_dict(saved["state_dict"])
restored_model.eval()

restored_scores = recon_err(restored_model, val_tensor)

assert np.allclose(
    restored_scores,
    reference,
    atol=1e-5,
    rtol=0
), "File checkpoint tải lại không khớp!"

print("\nPASS: Đã lưu và kiểm tra checkpoint thành công!")
print("Đường dẫn:", checkpoint_path.resolve())
print("MSE kiểm tra:", restored_scores.mean())


Seed: 42
Learning rate: 0.001
Best epoch: 1495
Best Validation MSE: 1.0372874736785889

PASS: Đã lưu và kiểm tra checkpoint thành công!
Đường dẫn: D:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks\outputs\06_2_ae_validation_A\06_2_ae_seed42_lr001_epoch1495_best.pt
MSE kiểm tra: 1.037287450534246
